# ⚡ TSRL — Optimization & Walk-Forward Analysis

Optimize strategy parameters using grid search and validate with walk-forward analysis.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

from src.application.services.data_service import DataService
from src.engine.backtest.engine import BacktestEngine, BacktestConfig
from src.engine.optimizer.optimizer import GridSearchOptimizer, OptimizationConfig
from src.engine.walkforward.walkforward import WalkForwardAnalysis
from src.strategies.momentum.ema_crossover import EMACrossoverStrategy

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Setup

In [ ]:
data_service = DataService()
df, source = data_service.fetch_data('AAPL', datetime(2020, 1, 1), datetime(2024, 12, 31))
print(f'Loaded {len(df)} bars ({source})')

config = BacktestConfig(
    initial_capital=100000,
    commission=0.001,
    slippage=0.0005,
)

## 2. Grid Search Optimization

In [ ]:
param_grid = {
    'fast_period': [5, 8, 10, 12, 15, 20],
    'slow_period': [20, 25, 30, 40, 50, 60],
}

opt_config = OptimizationConfig(metric='sharpe_ratio')
optimizer = GridSearchOptimizer(config=opt_config)

strategy = EMACrossoverStrategy()
result = optimizer.optimize(strategy, df, param_grid, config=config)

print(f'Best Score (Sharpe): {result.best_score:.4f}')
print(f'Best Params: {result.best_params}')
print(f'Total Combinations: {result.total_iterations}')
print(f'Time: {result.execution_time_ms:.0f}ms')

## 3. Parameter Heatmap

In [ ]:
# Build heatmap matrix
fast_vals = sorted(set(param_grid['fast_period']))
slow_vals = sorted(set(param_grid['slow_period']))

heatmap = np.full((len(slow_vals), len(fast_vals)), np.nan)

for r in result.all_results:
    if r.get('success'):
        fi = fast_vals.index(r['params']['fast_period'])
        si = slow_vals.index(r['params']['slow_period'])
        heatmap[si, fi] = r['score']

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(heatmap, cmap='RdYlGn', aspect='auto')

ax.set_xticks(range(len(fast_vals)))
ax.set_xticklabels(fast_vals)
ax.set_yticks(range(len(slow_vals)))
ax.set_yticklabels(slow_vals)
ax.set_xlabel('Fast Period', fontsize=12)
ax.set_ylabel('Slow Period', fontsize=12)
ax.set_title('Sharpe Ratio — Parameter Heatmap', fontsize=14, fontweight='bold')

# Annotate cells
for i in range(len(slow_vals)):
    for j in range(len(fast_vals)):
        if not np.isnan(heatmap[i, j]):
            color = 'white' if heatmap[i, j] < 0 else 'black'
            ax.text(j, i, f'{heatmap[i,j]:.2f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 4. Top Results

In [ ]:
top = sorted(
    [r for r in result.all_results if r.get('success')],
    key=lambda x: x['score'],
    reverse=True
)[:10]

top_df = pd.DataFrame([{
    'Fast': r['params']['fast_period'],
    'Slow': r['params']['slow_period'],
    'Sharpe': f"{r['score']:.4f}",
    'Return': f"{r.get('total_return', 0) * 100:.2f}%",
    'Trades': r.get('total_trades', 0),
    'Max DD': f"{r.get('max_drawdown', 0):.2f}%",
    'Win Rate': f"{r.get('win_rate', 0) * 100:.1f}%",
} for r in top])

top_df.index = range(1, len(top_df) + 1)
top_df.index.name = 'Rank'
top_df

## 5. Walk-Forward Analysis

In [ ]:
wfa = WalkForwardAnalysis()

wf_result = wfa.run(
    strategy_class=EMACrossoverStrategy,
    data=df,
    param_grid=param_grid,
    train_days=252,
    test_days=63,
    config=config,
)

print(f'Windows: {len(wf_result.windows)}')
print(f'Avg Train Sharpe: {wf_result.avg_train_sharpe:.4f}')
print(f'Avg Test Sharpe:  {wf_result.avg_test_sharpe:.4f}')
print(f'Stability Score:  {wf_result.stability_score:.4f}')
print(f'Total Test Return:{wf_result.total_test_return:.2%}')

## 6. Walk-Forward Windows

In [ ]:
windows_df = pd.DataFrame([{
    'Window': i + 1,
    'Train': f"{w.train_start.strftime('%Y-%m-%d')} → {w.train_end.strftime('%Y-%m-%d')}",
    'Test': f"{w.test_start.strftime('%Y-%m-%d')} → {w.test_end.strftime('%Y-%m-%d')}",
    'Best Params': str(w.best_params),
    'Test Return': f"{w.test_return:.2%}",
    'Trades': w.test_trades,
} for i, w in enumerate(wf_result.windows)])

windows_df.set_index('Window')

## 7. In-Sample vs Out-of-Sample

In [ ]:
if wf_result.windows:
    window_nums = list(range(1, len(wf_result.windows) + 1))
    test_returns = [w.test_return * 100 for w in wf_result.windows]

    fig, ax = plt.subplots(figsize=(12, 5))
    colors_bar = ['#69f0ae' if r >= 0 else '#ff5252' for r in test_returns]
    ax.bar(window_nums, test_returns, color=colors_bar, alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axhline(y=0, color='gray', linewidth=0.8)
    ax.set_xlabel('Walk-Forward Window')
    ax.set_ylabel('Out-of-Sample Return (%)')
    ax.set_title('Walk-Forward OOS Returns by Window', fontsize=14, fontweight='bold')
    ax.set_xticks(window_nums)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print('No walk-forward windows generated (need more data)')